## Revenue Forecasting

The revenue forecast uses a two-stage framework:

1. Forecast daily net ticket demand.
2. Forecast daily net revenue per ticket using only information available before the target date.
3. Multiply the two predictions to estimate daily net revenue.

The complete revenue forecast is evaluated through the same rolling 30-day
backtest used for demand forecasting. Actual future demand or revenue is never
used as a model feature.

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd

DATABASE_URL = "postgresql+psycopg:///orlando_demand_revenue"

engine = create_engine(DATABASE_URL)
revenue_query = text("""
    SELECT
        calendar_date,
        CASE
            WHEN has_actuals
                THEN net_revenue::double precision
            ELSE NULL
        END AS target_net_revenue,
        CASE
            WHEN has_actuals
                THEN (
                    net_revenue::double precision
                    / NULLIF(net_demand, 0)
                )
            ELSE NULL
        END AS target_net_revenue_per_ticket
    FROM analytics.vw_daily_performance
    ORDER BY calendar_date
""")

with engine.connect() as connection:
    revenue_targets = pd.read_sql_query(
        revenue_query,
        connection,
        parse_dates=["calendar_date"],
    )

query = text("""
    SELECT *
    FROM analytics.vw_forecast_input
    ORDER BY calendar_date
""")
with engine.connect() as connection:
    forecast_input = pd.read_sql_query(
        query,
        connection,
        parse_dates=["calendar_date"],
    )

forecast_input.shape
revenue_input = forecast_input.merge(
    revenue_targets,
    on="calendar_date",
    how="left",
    validate="one_to_one",
)

historical_revenue = revenue_input.loc[
    revenue_input["target_net_revenue"].notna()
].copy()

future_revenue = revenue_input.loc[
    revenue_input["target_net_revenue"].isna()
].copy()

assert len(revenue_input) == 1126
assert len(historical_revenue) == 1096
assert len(future_revenue) == 30

assert historical_revenue[
    [
        "target_net_revenue",
        "target_net_revenue_per_ticket",
    ]
].notna().all().all()

assert (
    historical_revenue["target_net_revenue"] > 0
).all()

assert (
    historical_revenue[
        "target_net_revenue_per_ticket"
    ] > 0
).all()

assert future_revenue[
    [
        "target_net_revenue",
        "target_net_revenue_per_ticket",
    ]
].isna().all().all()

historical_revenue[
    [
        "target_net_revenue",
        "target_net_revenue_per_ticket",
    ]
].describe().round(2)

,target_net_revenue,target_net_revenue_per_ticket
count,1096.00,1096.00
mean,155116.59,114.70
std,49779.29,6.99
min,70584.07,99.35
25%,115423.33,109.26
50%,146338.18,114.39
75%,184384.95,119.87
max,323896.73,136.03


In [2]:
product_revenue_query = text("""
    SELECT
        product.date_key,
        product.calendar_date,
        product.product_key,
        product.product_code,
        product.product_name,
        product.ticket_tier,
        product.base_price,

        product.day_of_week,
        product.month_number,
        product.year_number,
        product.is_weekend,
        product.holiday_flag,
        product.school_break_flag,
        product.season,

        product.planned_price_multiplier,
        product.planned_unit_list_price,

        forecast.active_campaign_count,
        forecast.maximum_planned_discount,
        forecast.paid_media_flag,
        forecast.email_campaign_flag,
        forecast.bundle_campaign_flag,
        forecast.discount_campaign_flag,

        product.net_demand
            AS target_product_demand,

        product.actual_product_demand_share
            AS target_product_share,

        product.net_revenue
            AS target_product_net_revenue,

        product.actual_net_revenue_per_ticket
            AS target_product_net_yield

    FROM analytics.vw_daily_product_performance AS product

    JOIN analytics.vw_forecast_input AS forecast
        USING (date_key)

    ORDER BY
        product.calendar_date,
        product.product_key
""")

with engine.connect() as connection:
    product_revenue_input = pd.read_sql_query(
        product_revenue_query,
        connection,
        parse_dates=["calendar_date"],
    )

historical_product_revenue = product_revenue_input.loc[
    product_revenue_input[
        "target_product_demand"
    ].notna()
].copy()

future_product_revenue = product_revenue_input.loc[
    product_revenue_input[
        "target_product_demand"
    ].isna()
].copy()

assert len(product_revenue_input) == 1126 * 4
assert len(historical_product_revenue) == 1096 * 4
assert len(future_product_revenue) == 30 * 4

assert historical_product_revenue.groupby(
    "calendar_date"
).size().eq(4).all()

assert future_product_revenue.groupby(
    "calendar_date"
).size().eq(4).all()

product_mix_summary = (
    historical_product_revenue
    .groupby(
        [
            "school_break_flag",
            "product_code",
        ]
    )
    .agg(
        net_demand=(
            "target_product_demand",
            "sum",
        )
    )
    .reset_index()
)

product_mix_summary["demand_share"] = (
    product_mix_summary["net_demand"]
    / product_mix_summary.groupby(
        "school_break_flag"
    )["net_demand"].transform("sum")
)

product_mix_summary.round(4)

,school_break_flag,product_code,net_demand,demand_share
0,False,ADULT_DAY,530290.0,0.5279
1,False,CHILD_DAY,249665.0,0.2485
2,False,EVENING,88823.0,0.0884
3,False,FLEX_DAY,135778.0,0.1352
4,True,ADULT_DAY,197353.0,0.4271
5,True,CHILD_DAY,178010.0,0.3853
6,True,EVENING,22675.0,0.0491
7,True,FLEX_DAY,64010.0,0.1385


### Load Out-of-Sample Demand Forecasts

Revenue models use the persisted SARIMAX out-of-sample demand predictions.
This prevents actual future demand from leaking into the revenue forecast.

In [5]:
import numpy as np
demand_backtest_query = text("""
    WITH latest_backtest_batch AS (
        SELECT MAX(created_at) AS created_at
        FROM analytics.fact_forecast
        WHERE forecast_run_type = 'backtest'
    )

    SELECT
        created_date.calendar_date
            AS forecast_created_date,

        target_date.calendar_date
            AS target_date,

        forecast.forecast_horizon_days
            AS horizon_days,

        forecast.predicted_demand::double precision
            AS predicted_demand,

        forecast.actual_demand::double precision
            AS actual_demand

    FROM analytics.fact_forecast AS forecast

    JOIN latest_backtest_batch AS latest
        ON forecast.created_at = latest.created_at

    JOIN analytics.dim_date AS created_date
        ON created_date.date_key
         = forecast.forecast_created_date_key

    JOIN analytics.dim_date AS target_date
        ON target_date.date_key
         = forecast.target_date_key

    WHERE forecast.forecast_run_type = 'backtest'
      AND forecast.model_name = 'sarimax_exogenous'

    ORDER BY
        forecast_created_date,
        target_date
""")

with engine.connect() as connection:
    sarimax_demand_backtests = pd.read_sql_query(
        demand_backtest_query,
        connection,
        parse_dates=[
            "forecast_created_date",
            "target_date",
        ],
    )

revenue_actuals = historical_revenue[
    [
        "calendar_date",
        "target_demand",
        "target_net_revenue",
        "target_net_revenue_per_ticket",
    ]
].rename(
    columns={
        "calendar_date": "target_date",
        "target_demand": "database_actual_demand",
        "target_net_revenue": "actual_net_revenue",
        "target_net_revenue_per_ticket": (
            "actual_net_revenue_per_ticket"
        ),
    }
)

revenue_backtest_base = (
    sarimax_demand_backtests
    .merge(
        revenue_actuals,
        on="target_date",
        how="left",
        validate="many_to_one",
    )
)

assert len(revenue_backtest_base) == 360

assert revenue_backtest_base[
    "forecast_created_date"
].nunique() == 12

assert revenue_backtest_base.groupby(
    "forecast_created_date"
).size().eq(30).all()

assert not revenue_backtest_base.duplicated(
    [
        "forecast_created_date",
        "target_date",
    ]
).any()

assert revenue_backtest_base[
    [
        "predicted_demand",
        "actual_demand",
        "actual_net_revenue",
    ]
].notna().all().all()

assert np.allclose(
    revenue_backtest_base["actual_demand"],
    revenue_backtest_base[
        "database_actual_demand"
    ],
)

revenue_backtest_base.head()

,forecast_created_date,target_date,horizon_days,predicted_demand,actual_demand,database_actual_demand,actual_net_revenue,actual_net_revenue_per_ticket
0,2024-12-31,2025-01-01,1,1382.75,1390.0,1390.0,173912.24,125.116719
1,2024-12-31,2025-01-02,2,1310.52,1496.0,1496.0,181600.96,121.391016
2,2024-12-31,2025-01-03,3,1538.47,1585.0,1585.0,187462.00,118.272555
3,2024-12-31,2025-01-04,4,1710.45,1382.0,1382.0,164157.61,118.782641
4,2024-12-31,2025-01-05,5,1495.73,1372.0,1372.0,158062.07,115.205590


### Revenue Baselines

Two revenue baselines are evaluated on the same 360 rolling-origin records:

1. Seasonal-naive revenue from the most recent available matching weekday.
2. SARIMAX predicted demand multiplied by the trailing 28-day weighted net
   revenue per ticket.

The weighted yield is calculated as total revenue divided by total net demand,
not as an unweighted average of daily ticket yields.

In [6]:
# Purpose:
# Calculate comparable daily revenue forecast metrics.
#
# Used by:
# Every revenue baseline and candidate model.
def calculate_revenue_metrics(predictions):
    actual = predictions[
        "actual_net_revenue"
    ].astype(float)

    predicted = predictions[
        "predicted_net_revenue"
    ].astype(float)

    error = predicted - actual
    nonzero_actual = actual.ne(0)

    return pd.Series({
        "observations": len(predictions),
        "mae": error.abs().mean(),
        "rmse": error.pow(2).mean() ** 0.5,
        "mape_pct": (
            error.loc[nonzero_actual].abs()
            / actual.loc[nonzero_actual]
        ).mean() * 100,
        "wape_pct": (
            error.abs().sum()
            / actual.abs().sum()
        ) * 100,
        "forecast_bias": error.mean(),
        "forecast_bias_pct": (
            error.sum() / actual.sum()
        ) * 100,
    })


backtest_origins = (
    revenue_backtest_base[
        "forecast_created_date"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

In [7]:
revenue_lookup = historical_revenue.set_index(
    "calendar_date"
)["target_net_revenue"]

seasonal_naive_revenue_folds = []

for cutoff in backtest_origins:
    fold = revenue_backtest_base.loc[
        revenue_backtest_base[
            "forecast_created_date"
        ].eq(cutoff)
    ].copy()

    weeks_back = (
        (fold["horizon_days"] - 1) // 7
    ) + 1

    fold["reference_date"] = (
        fold["target_date"]
        - pd.to_timedelta(
            weeks_back * 7,
            unit="D",
        )
    )

    fold["predicted_net_revenue"] = (
        fold["reference_date"].map(
            revenue_lookup
        )
    )

    fold["model_name"] = (
        "seasonal_naive_revenue_7d"
    )

    seasonal_naive_revenue_folds.append(
        fold
    )

seasonal_naive_revenue_predictions = (
    pd.concat(
        seasonal_naive_revenue_folds,
        ignore_index=True,
    )
)

assert len(
    seasonal_naive_revenue_predictions
) == 360

assert (
    seasonal_naive_revenue_predictions[
        "reference_date"
    ]
    <= seasonal_naive_revenue_predictions[
        "forecast_created_date"
    ]
).all()

assert seasonal_naive_revenue_predictions[
    "predicted_net_revenue"
].notna().all()

In [8]:
TRAILING_YIELD_DAYS = 28

aggregate_yield_folds = []

for cutoff in backtest_origins:
    training_window = (
        historical_revenue.loc[
            historical_revenue[
                "calendar_date"
            ].le(cutoff)
        ]
        .sort_values("calendar_date")
        .tail(TRAILING_YIELD_DAYS)
    )

    assert len(training_window) == (
        TRAILING_YIELD_DAYS
    )

    trailing_weighted_yield = (
        training_window[
            "target_net_revenue"
        ].sum()
        / training_window[
            "target_demand"
        ].sum()
    )

    fold = revenue_backtest_base.loc[
        revenue_backtest_base[
            "forecast_created_date"
        ].eq(cutoff)
    ].copy()

    fold[
        "predicted_net_revenue_per_ticket"
    ] = trailing_weighted_yield

    fold["predicted_net_revenue"] = (
        fold["predicted_demand"]
        * fold[
            "predicted_net_revenue_per_ticket"
        ]
    )

    fold["yield_training_start"] = (
        training_window[
            "calendar_date"
        ].min()
    )

    fold["yield_training_end"] = (
        training_window[
            "calendar_date"
        ].max()
    )

    fold["model_name"] = (
        "sarimax_demand_trailing_28d_yield"
    )

    aggregate_yield_folds.append(fold)

aggregate_yield_predictions = pd.concat(
    aggregate_yield_folds,
    ignore_index=True,
)

assert len(aggregate_yield_predictions) == 360

assert aggregate_yield_predictions[
    "predicted_net_revenue"
].notna().all()

assert (
    aggregate_yield_predictions[
        "yield_training_end"
    ]
    <= aggregate_yield_predictions[
        "forecast_created_date"
    ]
).all()

In [9]:
revenue_baseline_comparison = pd.DataFrame({
    "seasonal_naive_revenue_7d":
        calculate_revenue_metrics(
            seasonal_naive_revenue_predictions
        ),

    "sarimax_demand_trailing_28d_yield":
        calculate_revenue_metrics(
            aggregate_yield_predictions
        ),
}).T.sort_values("mae").round(2)

revenue_baseline_comparison

,observations,mae,rmse,mape_pct,wape_pct,forecast_bias,forecast_bias_pct
sarimax_demand_trailing_28d_yield,360.0,14999.46,20268.94,9.19,9.33,-1950.01,-1.21
seasonal_naive_revenue_7d,360.0,27318.45,36442.57,18.28,16.99,2812.89,1.75


### Product Mix Forecast

Product demand shares are forecast separately for each ticket product using only
information available at the forecast origin. Every model is evaluated on the
same 12 rolling origins and 360 target dates used by the demand and revenue
backtests.

Two approaches are compared:

1. Trailing 28-day realized product mix.
2. Ridge regression using product, calendar, planned price, and campaign features.

Predicted shares are clipped to nonnegative values and normalized to sum to one
for every target date. Product demand allocations therefore reconcile exactly
to the persisted SARIMAX total-demand forecast.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


PRODUCT_CATEGORICAL_FEATURES = [
    "product_code",
    "ticket_tier",
    "day_of_week",
    "month_number",
    "season",
]

PRODUCT_NUMERIC_FEATURES = [
    "trend_days",
    "base_price",
    "is_weekend",
    "holiday_flag",
    "school_break_flag",
    "planned_price_multiplier",
    "planned_unit_list_price",
    "active_campaign_count",
    "maximum_planned_discount",
    "paid_media_flag",
    "email_campaign_flag",
    "bundle_campaign_flag",
    "discount_campaign_flag",
]

PRODUCT_SHARE_ORIGIN_DATE = (
    product_revenue_input["calendar_date"].min()
)


# Purpose:
# Convert product-date records into model-ready features.
#
# Used by:
# Every product-share model in the rolling-origin backtest.
def prepare_product_share_features(frame):
    features = frame.copy()

    features["trend_days"] = (
        features["calendar_date"]
        - PRODUCT_SHARE_ORIGIN_DATE
    ).dt.days

    for column in PRODUCT_CATEGORICAL_FEATURES:
        features[column] = features[column].astype(str)

    for column in PRODUCT_NUMERIC_FEATURES:
        features[column] = (
            pd.to_numeric(
                features[column],
                errors="raise",
            )
            .astype(float)
        )

    return features[
        PRODUCT_CATEGORICAL_FEATURES
        + PRODUCT_NUMERIC_FEATURES
    ]


# Purpose:
# Create a new leakage-safe Ridge product-share model.
#
# Used by:
# Each rolling-origin fold so fitted state is never shared across folds.
def build_product_share_model():
    preprocessing = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                PRODUCT_CATEGORICAL_FEATURES,
            ),
            (
                "numeric",
                StandardScaler(),
                PRODUCT_NUMERIC_FEATURES,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("model", Ridge(alpha=1.0)),
        ]
    )


# Purpose:
# Force product predictions to be valid compositional shares.
#
# Used by:
# Product-share baselines and candidate models before demand allocation.
def normalize_product_shares(frame, raw_predictions):
    normalized = frame.copy()

    normalized["predicted_product_share"] = np.clip(
        np.asarray(raw_predictions, dtype=float),
        1e-6,
        None,
    )

    share_totals = normalized.groupby(
        [
            "forecast_created_date",
            "target_date",
        ]
    )["predicted_product_share"].transform("sum")

    normalized["predicted_product_share"] = (
        normalized["predicted_product_share"]
        / share_totals
    )

    normalized["predicted_product_demand"] = (
        normalized["predicted_demand"]
        * normalized["predicted_product_share"]
    )

    return normalized

In [11]:
TRAILING_PRODUCT_MIX_DAYS = 28

trailing_product_mix_folds = []
ridge_product_mix_folds = []

for cutoff in backtest_origins:
    training_data = (
        historical_product_revenue.loc[
            historical_product_revenue[
                "calendar_date"
            ].le(cutoff)
        ]
        .sort_values(
            [
                "calendar_date",
                "product_key",
            ]
        )
        .copy()
    )

    fold_demand = revenue_backtest_base.loc[
        revenue_backtest_base[
            "forecast_created_date"
        ].eq(cutoff),
        [
            "target_date",
            "horizon_days",
            "predicted_demand",
            "actual_demand",
        ],
    ].copy()

    scoring_data = (
        product_revenue_input.loc[
            product_revenue_input[
                "calendar_date"
            ].isin(fold_demand["target_date"])
        ]
        .copy()
        .rename(
            columns={
                "calendar_date": "target_date",
            }
        )
        .merge(
            fold_demand,
            on="target_date",
            how="inner",
            validate="many_to_one",
        )
    )

    scoring_data["forecast_created_date"] = cutoff

    # Baseline: realized mix from the last 28 available dates.
    trailing_dates = (
        training_data["calendar_date"]
        .drop_duplicates()
        .sort_values()
        .tail(TRAILING_PRODUCT_MIX_DAYS)
    )

    trailing_window = training_data.loc[
        training_data["calendar_date"].isin(
            trailing_dates
        )
    ]

    trailing_mix = (
        trailing_window.groupby(
            "product_code"
        )["target_product_demand"].sum()
        / trailing_window[
            "target_product_demand"
        ].sum()
    )

    trailing_fold = scoring_data.copy()

    trailing_raw_predictions = (
        trailing_fold["product_code"]
        .map(trailing_mix)
        .astype(float)
        .to_numpy()
    )

    trailing_fold = normalize_product_shares(
        trailing_fold,
        trailing_raw_predictions,
    )

    trailing_fold["model_name"] = (
        "trailing_28d_product_mix"
    )

    trailing_product_mix_folds.append(
        trailing_fold
    )

    # Candidate: calendar-, price-, and campaign-aware Ridge model.
    ridge_model = build_product_share_model()

    ridge_model.fit(
        prepare_product_share_features(
            training_data
        ),
        training_data[
            "target_product_share"
        ].astype(float),
    )

    scoring_features = scoring_data.rename(
        columns={
            "target_date": "calendar_date",
        }
    )

    ridge_raw_predictions = ridge_model.predict(
        prepare_product_share_features(
            scoring_features
        )
    )

    ridge_fold = normalize_product_shares(
        scoring_data,
        ridge_raw_predictions,
    )

    ridge_fold["model_name"] = (
        "ridge_product_mix"
    )

    ridge_product_mix_folds.append(
        ridge_fold
    )


trailing_product_mix_predictions = pd.concat(
    trailing_product_mix_folds,
    ignore_index=True,
)

ridge_product_mix_predictions = pd.concat(
    ridge_product_mix_folds,
    ignore_index=True,
)

for predictions in [
    trailing_product_mix_predictions,
    ridge_product_mix_predictions,
]:
    assert len(predictions) == 360 * 4

    predicted_share_totals = predictions.groupby(
        [
            "forecast_created_date",
            "target_date",
        ]
    )["predicted_product_share"].sum()

    assert np.allclose(
        predicted_share_totals,
        1.0,
    )

    allocated_demand = predictions.groupby(
        [
            "forecast_created_date",
            "target_date",
        ]
    ).agg(
        allocated_product_demand=(
            "predicted_product_demand",
            "sum",
        ),
        predicted_total_demand=(
            "predicted_demand",
            "first",
        ),
    )

    assert np.allclose(
        allocated_demand[
            "allocated_product_demand"
        ],
        allocated_demand[
            "predicted_total_demand"
        ],
    )

In [12]:
# Purpose:
# Evaluate both share accuracy and allocated product-demand accuracy.
#
# Used by:
# Product-mix model comparison and final revenue model selection.
def calculate_product_mix_metrics(predictions):
    share_error = (
        predictions["predicted_product_share"]
        - predictions[
            "target_product_share"
        ].astype(float)
    )

    product_demand_error = (
        predictions["predicted_product_demand"]
        - predictions[
            "target_product_demand"
        ].astype(float)
    )

    daily_mix_distance = (
        predictions.assign(
            absolute_share_error=share_error.abs()
        )
        .groupby(
            [
                "forecast_created_date",
                "target_date",
            ]
        )["absolute_share_error"]
        .sum()
        * 0.5
    )

    return pd.Series({
        "product_date_observations": len(
            predictions
        ),
        "share_mae_percentage_points": (
            share_error.abs().mean() * 100
        ),
        "average_daily_mix_distance_pct": (
            daily_mix_distance.mean() * 100
        ),
        "product_demand_mae": (
            product_demand_error.abs().mean()
        ),
        "product_demand_rmse": (
            product_demand_error.pow(2).mean()
            ** 0.5
        ),
    })


product_mix_comparison = pd.DataFrame({
    "trailing_28d_product_mix":
        calculate_product_mix_metrics(
            trailing_product_mix_predictions
        ),

    "ridge_product_mix":
        calculate_product_mix_metrics(
            ridge_product_mix_predictions
        ),
}).T.sort_values(
    "average_daily_mix_distance_pct"
).round(3)

product_mix_comparison

,product_date_observations,share_mae_percentage_points,average_daily_mix_distance_pct,product_demand_mae,product_demand_rmse
trailing_28d_product_mix,1440.0,3.301,6.602,53.586,76.171
ridge_product_mix,1440.0,3.624,7.247,57.387,82.300


### Nonlinear Product Mix Model

Ridge regression did not outperform the trailing 28-day product-mix baseline.
A histogram gradient boosting model is therefore evaluated using the same
features, forecast origins, and target dates. This model can capture nonlinear
effects and interactions between product type, calendar conditions, planned
pricing, and campaigns.

In [13]:
from sklearn.ensemble import HistGradientBoostingRegressor


# Purpose:
# Create a nonlinear product-share model with fixed hyperparameters.
#
# Used by:
# The rolling-origin product-mix backtest.
def build_hgb_product_share_model():
    preprocessing = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                PRODUCT_CATEGORICAL_FEATURES,
            ),
            (
                "numeric",
                StandardScaler(),
                PRODUCT_NUMERIC_FEATURES,
            ),
        ]
    )

    model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    )

    return Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("model", model),
        ]
    )


# Purpose:
# Evaluate one product-share model across all 12 rolling origins.
#
# Used by:
# HGB and any later product-mix candidate models.
def run_product_share_backtest(
    model_builder,
    model_name,
):
    prediction_folds = []

    for cutoff in backtest_origins:
        training_data = (
            historical_product_revenue.loc[
                historical_product_revenue[
                    "calendar_date"
                ].le(cutoff)
            ]
            .sort_values(
                [
                    "calendar_date",
                    "product_key",
                ]
            )
            .copy()
        )

        fold_demand = revenue_backtest_base.loc[
            revenue_backtest_base[
                "forecast_created_date"
            ].eq(cutoff),
            [
                "target_date",
                "horizon_days",
                "predicted_demand",
                "actual_demand",
            ],
        ].copy()

        scoring_data = (
            product_revenue_input.loc[
                product_revenue_input[
                    "calendar_date"
                ].isin(fold_demand["target_date"])
            ]
            .copy()
            .rename(
                columns={
                    "calendar_date": "target_date",
                }
            )
            .merge(
                fold_demand,
                on="target_date",
                how="inner",
                validate="many_to_one",
            )
        )

        scoring_data[
            "forecast_created_date"
        ] = cutoff

        model = model_builder()

        model.fit(
            prepare_product_share_features(
                training_data
            ),
            training_data[
                "target_product_share"
            ].astype(float),
        )

        scoring_features = scoring_data.rename(
            columns={
                "target_date": "calendar_date",
            }
        )

        raw_predictions = model.predict(
            prepare_product_share_features(
                scoring_features
            )
        )

        fold_predictions = (
            normalize_product_shares(
                scoring_data,
                raw_predictions,
            )
        )

        fold_predictions["model_name"] = (
            model_name
        )

        prediction_folds.append(
            fold_predictions
        )

    predictions = pd.concat(
        prediction_folds,
        ignore_index=True,
    )

    assert len(predictions) == 360 * 4

    share_totals = predictions.groupby(
        [
            "forecast_created_date",
            "target_date",
        ]
    )["predicted_product_share"].sum()

    assert np.allclose(
        share_totals,
        1.0,
    )

    allocated_demand = predictions.groupby(
        [
            "forecast_created_date",
            "target_date",
        ]
    ).agg(
        allocated_product_demand=(
            "predicted_product_demand",
            "sum",
        ),
        predicted_total_demand=(
            "predicted_demand",
            "first",
        ),
    )

    assert np.allclose(
        allocated_demand[
            "allocated_product_demand"
        ],
        allocated_demand[
            "predicted_total_demand"
        ],
    )

    return predictions

In [14]:
hgb_product_mix_predictions = (
    run_product_share_backtest(
        model_builder=(
            build_hgb_product_share_model
        ),
        model_name="hgb_product_mix",
    )
)

product_mix_comparison = pd.DataFrame({
    "trailing_28d_product_mix":
        calculate_product_mix_metrics(
            trailing_product_mix_predictions
        ),

    "ridge_product_mix":
        calculate_product_mix_metrics(
            ridge_product_mix_predictions
        ),

    "hgb_product_mix":
        calculate_product_mix_metrics(
            hgb_product_mix_predictions
        ),
}).T.sort_values(
    "average_daily_mix_distance_pct"
).round(3)

product_mix_comparison

,product_date_observations,share_mae_percentage_points,average_daily_mix_distance_pct,product_demand_mae,product_demand_rmse
hgb_product_mix,1440.0,1.712,3.423,35.524,51.295
trailing_28d_product_mix,1440.0,3.301,6.602,53.586,76.171
ridge_product_mix,1440.0,3.624,7.247,57.387,82.300


### Bottom-Up Revenue with Trailing Product Yields

The selected HGB product-mix model allocates the SARIMAX total-demand forecast
across the four ticket products. For each forecast origin, every product's net
revenue per ticket is estimated from its most recent 28 available days.

Product-level revenue is calculated as:

$$
\widehat{R}_{t,p}
=
\widehat{D}_{t}
\times
\widehat{s}_{t,p}
\times
\widehat{Y}_{p,28}
$$

Daily revenue is the sum of the four product forecasts. This isolates whether
the more accurate product-mix forecast improves revenue before a separate
product-yield model is introduced.

In [15]:
TRAILING_PRODUCT_YIELD_DAYS = 28

bottom_up_trailing_yield_folds = []

for cutoff in backtest_origins:
    available_history = (
        historical_product_revenue.loc[
            historical_product_revenue[
                "calendar_date"
            ].le(cutoff)
        ]
        .sort_values(
            [
                "calendar_date",
                "product_key",
            ]
        )
        .copy()
    )

    trailing_dates = (
        available_history["calendar_date"]
        .drop_duplicates()
        .sort_values()
        .tail(TRAILING_PRODUCT_YIELD_DAYS)
    )

    yield_window = available_history.loc[
        available_history["calendar_date"].isin(
            trailing_dates
        )
    ].copy()

    assert (
        yield_window["calendar_date"].nunique()
        == TRAILING_PRODUCT_YIELD_DAYS
    )

    product_yields = (
        yield_window.groupby(
            [
                "product_key",
                "product_code",
            ],
            as_index=False,
        )
        .agg(
            trailing_net_revenue=(
                "target_product_net_revenue",
                "sum",
            ),
            trailing_net_demand=(
                "target_product_demand",
                "sum",
            ),
        )
    )

    product_yields[
        "predicted_product_net_yield"
    ] = (
        product_yields["trailing_net_revenue"]
        / product_yields["trailing_net_demand"]
    )

    assert (
        product_yields[
            "predicted_product_net_yield"
        ] > 0
    ).all()

    fold = hgb_product_mix_predictions.loc[
        hgb_product_mix_predictions[
            "forecast_created_date"
        ].eq(cutoff)
    ].copy()

    fold = fold.merge(
        product_yields[
            [
                "product_key",
                "predicted_product_net_yield",
            ]
        ],
        on="product_key",
        how="left",
        validate="many_to_one",
    )

    assert fold[
        "predicted_product_net_yield"
    ].notna().all()

    fold["predicted_product_net_revenue"] = (
        fold["predicted_product_demand"]
        * fold["predicted_product_net_yield"]
    )

    fold["yield_training_start"] = (
        yield_window["calendar_date"].min()
    )

    fold["yield_training_end"] = (
        yield_window["calendar_date"].max()
    )

    assert (
        fold["yield_training_end"] <= cutoff
    ).all()

    bottom_up_trailing_yield_folds.append(fold)


bottom_up_product_predictions = pd.concat(
    bottom_up_trailing_yield_folds,
    ignore_index=True,
)

assert len(bottom_up_product_predictions) == 360 * 4

bottom_up_revenue_predictions = (
    bottom_up_product_predictions.groupby(
        [
            "forecast_created_date",
            "target_date",
            "horizon_days",
        ],
        as_index=False,
    )
    .agg(
        predicted_net_revenue=(
            "predicted_product_net_revenue",
            "sum",
        ),
        actual_net_revenue=(
            "target_product_net_revenue",
            "sum",
        ),
        predicted_demand=(
            "predicted_demand",
            "first",
        ),
        actual_demand=(
            "actual_demand",
            "first",
        ),
    )
)

bottom_up_revenue_predictions["model_name"] = (
    "sarimax_demand_hgb_mix_trailing_product_yield"
)

assert len(bottom_up_revenue_predictions) == 360

assert bottom_up_revenue_predictions.groupby(
    "forecast_created_date"
).size().eq(30).all()

revenue_reconciliation = (
    bottom_up_revenue_predictions.merge(
        revenue_backtest_base[
            [
                "forecast_created_date",
                "target_date",
                "actual_net_revenue",
            ]
        ],
        on=[
            "forecast_created_date",
            "target_date",
        ],
        how="left",
        suffixes=(
            "_product_sum",
            "_daily_table",
        ),
        validate="one_to_one",
    )
)

assert np.allclose(
    revenue_reconciliation[
        "actual_net_revenue_product_sum"
    ],
    revenue_reconciliation[
        "actual_net_revenue_daily_table"
    ],
    atol=0.01,
)

In [16]:
revenue_model_comparison = pd.DataFrame({
    "seasonal_naive_revenue_7d":
        calculate_revenue_metrics(
            seasonal_naive_revenue_predictions
        ),

    "sarimax_demand_trailing_28d_yield":
        calculate_revenue_metrics(
            aggregate_yield_predictions
        ),

    "sarimax_demand_hgb_mix_trailing_product_yield":
        calculate_revenue_metrics(
            bottom_up_revenue_predictions
        ),
}).T.sort_values(
    "mae"
).round(2)

revenue_model_comparison

,observations,mae,rmse,mape_pct,wape_pct,forecast_bias,forecast_bias_pct
sarimax_demand_hgb_mix_trailing_product_yield,360.0,14681.28,19767.77,8.98,9.13,-1850.96,-1.15
sarimax_demand_trailing_28d_yield,360.0,14999.46,20268.94,9.19,9.33,-1950.01,-1.21
seasonal_naive_revenue_7d,360.0,27318.45,36442.57,18.28,16.99,2812.89,1.75


### Product-Level Net Yield Models

The SARIMAX total-demand forecast and selected HGB product-mix forecast are held
constant. This stage compares alternative methods for forecasting each
product's net revenue per ticket:

1. Trailing 28-day weighted product yield.
2. Ridge regression.
3. Histogram gradient boosting.

All yield models use only data available at each forecast origin. Their
product-level revenue forecasts are summed to daily revenue and evaluated
against the same 360 origin-target observations.

In [21]:
# Purpose:
# Create an interpretable linear product-yield model.
#
# Used by:
# The rolling-origin product-yield backtest.
def build_ridge_product_yield_model():
    preprocessing = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                PRODUCT_CATEGORICAL_FEATURES,
            ),
            (
                "numeric",
                StandardScaler(),
                PRODUCT_NUMERIC_FEATURES,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("model", Ridge(alpha=1.0)),
        ]
    )


# Purpose:
# Create a nonlinear product-yield model with fixed parameters.
#
# Used by:
# The rolling-origin product-yield backtest.
def build_hgb_product_yield_model():
    preprocessing = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                PRODUCT_CATEGORICAL_FEATURES,
            ),
            (
                "numeric",
                StandardScaler(),
                PRODUCT_NUMERIC_FEATURES,
            ),
        ]
    )

    model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    )

    return Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("model", model),
        ]
    )

In [18]:
# Purpose:
# Forecast product net revenue per ticket across all rolling origins.
#
# Used by:
# Ridge and HGB yield evaluation and bottom-up revenue calculation.
def run_product_yield_backtest(
    model_builder,
    model_name,
):
    prediction_folds = []

    for cutoff in backtest_origins:
        training_data = (
            historical_product_revenue.loc[
                (
                    historical_product_revenue[
                        "calendar_date"
                    ].le(cutoff)
                )
                & (
                    historical_product_revenue[
                        "target_product_net_yield"
                    ].notna()
                )
            ]
            .sort_values(
                [
                    "calendar_date",
                    "product_key",
                ]
            )
            .copy()
        )

        scoring_data = (
            hgb_product_mix_predictions.loc[
                hgb_product_mix_predictions[
                    "forecast_created_date"
                ].eq(cutoff)
            ]
            .sort_values(
                [
                    "target_date",
                    "product_key",
                ]
            )
            .copy()
        )

        model = model_builder()

        model.fit(
            prepare_product_share_features(
                training_data
            ),
            training_data[
                "target_product_net_yield"
            ].astype(float),
        )

        scoring_features = scoring_data.rename(
            columns={
                "target_date": "calendar_date",
            }
        )

        predicted_yield = model.predict(
            prepare_product_share_features(
                scoring_features
            )
        )

        scoring_data[
            "predicted_product_net_yield"
        ] = np.maximum(
            predicted_yield,
            0.01,
        )

        scoring_data["yield_model_name"] = (
            model_name
        )

        scoring_data["yield_training_end"] = (
            training_data["calendar_date"].max()
        )

        assert (
            scoring_data["yield_training_end"]
            <= cutoff
        ).all()

        prediction_folds.append(scoring_data)

    predictions = pd.concat(
        prediction_folds,
        ignore_index=True,
    )

    assert len(predictions) == 360 * 4

    assert predictions[
        "predicted_product_net_yield"
    ].notna().all()

    assert (
        predictions[
            "predicted_product_net_yield"
        ] > 0
    ).all()

    return predictions


ridge_product_yield_predictions = (
    run_product_yield_backtest(
        model_builder=(
            build_ridge_product_yield_model
        ),
        model_name="ridge_product_yield",
    )
)

hgb_product_yield_predictions = (
    run_product_yield_backtest(
        model_builder=(
            build_hgb_product_yield_model
        ),
        model_name="hgb_product_yield",
    )
)

In [19]:
# Purpose:
# Evaluate product-level yield predictions before revenue aggregation.
#
# Used by:
# Product-yield model selection.
def calculate_product_yield_metrics(
    predictions,
):
    actual = predictions[
        "target_product_net_yield"
    ].astype(float)

    predicted = predictions[
        "predicted_product_net_yield"
    ].astype(float)

    demand_weights = predictions[
        "target_product_demand"
    ].astype(float)

    error = predicted - actual
    nonzero_actual = actual.ne(0)

    return pd.Series({
        "observations": len(predictions),

        "yield_mae": (
            error.abs().mean()
        ),

        "yield_rmse": (
            error.pow(2).mean() ** 0.5
        ),

        "yield_mape_pct": (
            (
                error.loc[nonzero_actual].abs()
                / actual.loc[nonzero_actual]
            ).mean()
            * 100
        ),

        "demand_weighted_yield_mae": (
            (
                error.abs()
                * demand_weights
            ).sum()
            / demand_weights.sum()
        ),

        "yield_bias": error.mean(),
    })


product_yield_comparison = pd.DataFrame({
    "trailing_28d_product_yield":
        calculate_product_yield_metrics(
            bottom_up_product_predictions
        ),

    "ridge_product_yield":
        calculate_product_yield_metrics(
            ridge_product_yield_predictions
        ),

    "hgb_product_yield":
        calculate_product_yield_metrics(
            hgb_product_yield_predictions
        ),
}).T.sort_values(
    "demand_weighted_yield_mae"
).round(3)

product_yield_comparison

,observations,yield_mae,yield_rmse,yield_mape_pct,demand_weighted_yield_mae,yield_bias
ridge_product_yield,1440.0,0.524,0.688,0.492,0.424,-0.024
hgb_product_yield,1440.0,0.509,0.690,0.465,0.438,-0.029
trailing_28d_product_yield,1440.0,5.566,6.924,4.965,5.642,0.355


In [22]:
# Purpose:
# Convert product-demand and product-yield forecasts into daily revenue.
#
# Used by:
# Final bottom-up revenue model comparison.
def aggregate_product_revenue(
    product_predictions,
    model_name,
):
    predictions = product_predictions.copy()

    predictions[
        "predicted_product_net_revenue"
    ] = (
        predictions[
            "predicted_product_demand"
        ]
        * predictions[
            "predicted_product_net_yield"
        ]
    )

    daily = (
        predictions.groupby(
            [
                "forecast_created_date",
                "target_date",
                "horizon_days",
            ],
            as_index=False,
        )
        .agg(
            predicted_net_revenue=(
                "predicted_product_net_revenue",
                "sum",
            ),
            actual_net_revenue=(
                "target_product_net_revenue",
                "sum",
            ),
            predicted_demand=(
                "predicted_demand",
                "first",
            ),
            actual_demand=(
                "actual_demand",
                "first",
            ),
        )
    )

    daily["model_name"] = model_name

    assert len(daily) == 360

    assert daily.groupby(
        "forecast_created_date"
    ).size().eq(30).all()

    return daily


ridge_yield_revenue_predictions = (
    aggregate_product_revenue(
        ridge_product_yield_predictions,
        (
            "sarimax_demand_hgb_mix_"
            "ridge_product_yield"
        ),
    )
)

hgb_yield_revenue_predictions = (
    aggregate_product_revenue(
        hgb_product_yield_predictions,
        (
            "sarimax_demand_hgb_mix_"
            "hgb_product_yield"
        ),
    )
)


revenue_model_comparison = pd.DataFrame({
    "seasonal_naive_revenue_7d":
        calculate_revenue_metrics(
            seasonal_naive_revenue_predictions
        ),

    "sarimax_demand_trailing_28d_yield":
        calculate_revenue_metrics(
            aggregate_yield_predictions
        ),

    "sarimax_demand_hgb_mix_trailing_product_yield":
        calculate_revenue_metrics(
            bottom_up_revenue_predictions
        ),

    "sarimax_demand_hgb_mix_ridge_product_yield":
        calculate_revenue_metrics(
            ridge_yield_revenue_predictions
        ),

    "sarimax_demand_hgb_mix_hgb_product_yield":
        calculate_revenue_metrics(
            hgb_yield_revenue_predictions
        ),
}).T.sort_values(
    "mae"
).round(2)

revenue_model_comparison

,observations,mae,rmse,mape_pct,wape_pct,forecast_bias,forecast_bias_pct
sarimax_demand_hgb_mix_ridge_product_yield,360.0,12085.33,16435.20,7.54,7.51,-953.68,-0.59
sarimax_demand_hgb_mix_hgb_product_yield,360.0,12113.09,16478.33,7.54,7.53,-988.89,-0.61
sarimax_demand_hgb_mix_trailing_product_yield,360.0,14681.28,19767.77,8.98,9.13,-1850.96,-1.15
sarimax_demand_trailing_28d_yield,360.0,14999.46,20268.94,9.19,9.33,-1950.01,-1.21
seasonal_naive_revenue_7d,360.0,27318.45,36442.57,18.28,16.99,2812.89,1.75


### Revenue Forecast Intervals

Revenue intervals are calibrated from end-to-end errors produced by the selected
SARIMAX demand, HGB product-mix, and Ridge product-yield pipeline.

An 80% time-ordered conformal interval is calculated separately for four
forecast-horizon buckets. During backtesting, each forecast origin may use only
errors from earlier origins, preventing future leakage. The first three origins
are retained only as initial calibration observations.

In [23]:
import math


REVENUE_INTERVAL_ALPHA = 0.20

REVENUE_HORIZON_LABELS = [
    "Days 1-7",
    "Days 8-14",
    "Days 15-21",
    "Days 22-30",
]


# Purpose:
# Calculate a finite-sample conformal absolute-error quantile.
#
# Used by:
# Revenue backtest and production interval calibration.
def calculate_conformal_quantile(
    absolute_errors,
    alpha=REVENUE_INTERVAL_ALPHA,
):
    scores = np.sort(
        np.asarray(
            absolute_errors,
            dtype=float,
        )
    )

    if len(scores) == 0:
        raise ValueError(
            "Conformal calibration requires errors."
        )

    rank = min(
        math.ceil(
            (len(scores) + 1)
            * (1 - alpha)
        ),
        len(scores),
    )

    return float(scores[rank - 1])

In [24]:
revenue_interval_backtest = (
    ridge_yield_revenue_predictions.copy()
)

revenue_interval_backtest[
    "horizon_bucket"
] = pd.cut(
    revenue_interval_backtest["horizon_days"],
    bins=[0, 7, 14, 21, 30],
    labels=REVENUE_HORIZON_LABELS,
)

revenue_interval_backtest[
    "absolute_error"
] = (
    revenue_interval_backtest[
        "actual_net_revenue"
    ]
    - revenue_interval_backtest[
        "predicted_net_revenue"
    ]
).abs()

revenue_interval_backtest["lower_bound"] = np.nan
revenue_interval_backtest["upper_bound"] = np.nan
revenue_interval_backtest[
    "interval_confidence"
] = np.nan
revenue_interval_backtest[
    "calibration_observations"
] = np.nan

revenue_origins = sorted(
    revenue_interval_backtest[
        "forecast_created_date"
    ].unique()
)

MINIMUM_REVENUE_CALIBRATION_ORIGINS = 3

for origin_number, origin in enumerate(
    revenue_origins
):
    if (
        origin_number
        < MINIMUM_REVENUE_CALIBRATION_ORIGINS
    ):
        continue

    prior_predictions = (
        revenue_interval_backtest.loc[
            revenue_interval_backtest[
                "forecast_created_date"
            ].lt(origin)
        ]
    )

    for horizon_bucket in (
        REVENUE_HORIZON_LABELS
    ):
        calibration_errors = (
            prior_predictions.loc[
                prior_predictions[
                    "horizon_bucket"
                ].eq(horizon_bucket),
                "absolute_error",
            ]
        )

        quantile = calculate_conformal_quantile(
            calibration_errors
        )

        target_rows = (
            revenue_interval_backtest[
                "forecast_created_date"
            ].eq(origin)
            & revenue_interval_backtest[
                "horizon_bucket"
            ].eq(horizon_bucket)
        )

        point_forecast = (
            revenue_interval_backtest.loc[
                target_rows,
                "predicted_net_revenue",
            ]
        )

        revenue_interval_backtest.loc[
            target_rows,
            "lower_bound",
        ] = np.maximum(
            point_forecast - quantile,
            0,
        )

        revenue_interval_backtest.loc[
            target_rows,
            "upper_bound",
        ] = point_forecast + quantile

        revenue_interval_backtest.loc[
            target_rows,
            "interval_confidence",
        ] = 1 - REVENUE_INTERVAL_ALPHA

        revenue_interval_backtest.loc[
            target_rows,
            "calibration_observations",
        ] = len(calibration_errors)

In [25]:
evaluated_revenue_intervals = (
    revenue_interval_backtest.dropna(
        subset=[
            "lower_bound",
            "upper_bound",
        ]
    )
    .copy()
)

evaluated_revenue_intervals["covered"] = (
    evaluated_revenue_intervals[
        "actual_net_revenue"
    ].between(
        evaluated_revenue_intervals[
            "lower_bound"
        ],
        evaluated_revenue_intervals[
            "upper_bound"
        ],
        inclusive="both",
    )
)

evaluated_revenue_intervals[
    "interval_width"
] = (
    evaluated_revenue_intervals[
        "upper_bound"
    ]
    - evaluated_revenue_intervals[
        "lower_bound"
    ]
)

revenue_interval_summary = (
    evaluated_revenue_intervals.groupby(
        "horizon_bucket",
        observed=True,
    )
    .agg(
        observations=(
            "covered",
            "size",
        ),
        coverage_pct=(
            "covered",
            lambda values: values.mean() * 100,
        ),
        average_interval_width=(
            "interval_width",
            "mean",
        ),
        average_point_forecast=(
            "predicted_net_revenue",
            "mean",
        ),
    )
)

revenue_interval_summary[
    "average_width_pct_of_forecast"
] = (
    revenue_interval_summary[
        "average_interval_width"
    ]
    / revenue_interval_summary[
        "average_point_forecast"
    ]
    * 100
)

revenue_interval_summary.round(2)

,observations,coverage_pct,average_interval_width,average_point_forecast,average_width_pct_of_forecast
horizon_bucket,,,,,
Days 1-7,63,82.54,48618.40,162922.47,29.84
Days 8-14,63,77.78,32737.85,159783.38,20.49
Days 15-21,63,84.13,47152.61,157183.79,30.00
Days 22-30,81,87.65,42066.20,165387.20,25.43


## Production Forecast

In [26]:
production_demand_query = text("""
    WITH latest_production_batch AS (
        SELECT MAX(created_at) AS created_at
        FROM analytics.fact_forecast
        WHERE forecast_run_type = 'production'
          AND model_name = 'sarimax_exogenous'
    )

    SELECT
        created_date.calendar_date
            AS forecast_created_date,

        target_date.calendar_date
            AS target_date,

        forecast.forecast_horizon_days
            AS horizon_days,

        forecast.predicted_demand::double precision
            AS predicted_demand

    FROM analytics.fact_forecast AS forecast

    JOIN latest_production_batch AS latest
        ON forecast.created_at = latest.created_at

    JOIN analytics.dim_date AS created_date
        ON created_date.date_key
         = forecast.forecast_created_date_key

    JOIN analytics.dim_date AS target_date
        ON target_date.date_key
         = forecast.target_date_key

    WHERE forecast.forecast_run_type = 'production'
      AND forecast.model_name = 'sarimax_exogenous'

    ORDER BY target_date
""")

with engine.connect() as connection:
    production_demand_forecast = (
        pd.read_sql_query(
            production_demand_query,
            connection,
            parse_dates=[
                "forecast_created_date",
                "target_date",
            ],
        )
    )

assert len(production_demand_forecast) == 30

assert production_demand_forecast[
    "predicted_demand"
].notna().all()

production_demand_forecast.head()

,forecast_created_date,target_date,horizon_days,predicted_demand
0,2025-12-31,2026-01-01,1,1591.95
1,2025-12-31,2026-01-02,2,1572.14
2,2025-12-31,2026-01-03,3,1750.00
3,2025-12-31,2026-01-04,4,1515.34
4,2025-12-31,2026-01-05,5,1004.06


In [27]:
production_mix_model = (
    build_hgb_product_share_model()
)

production_mix_model.fit(
    prepare_product_share_features(
        historical_product_revenue
    ),
    historical_product_revenue[
        "target_product_share"
    ].astype(float),
)

future_product_scoring = (
    future_product_revenue
    .rename(
        columns={
            "calendar_date": "target_date",
        }
    )
    .merge(
        production_demand_forecast,
        on="target_date",
        how="inner",
        validate="many_to_one",
    )
)

future_mix_features = (
    future_product_scoring.rename(
        columns={
            "target_date": "calendar_date",
        }
    )
)

future_raw_product_shares = (
    production_mix_model.predict(
        prepare_product_share_features(
            future_mix_features
        )
    )
)

future_product_predictions = (
    normalize_product_shares(
        future_product_scoring,
        future_raw_product_shares,
    )
)

future_product_predictions[
    "product_mix_model_name"
] = "hgb_product_mix"

assert len(future_product_predictions) == 30 * 4

assert np.allclose(
    future_product_predictions.groupby(
        "target_date"
    )["predicted_product_share"].sum(),
    1.0,
)

assert np.allclose(
    future_product_predictions.groupby(
        "target_date"
    )["predicted_product_demand"].sum(),
    production_demand_forecast.set_index(
        "target_date"
    )["predicted_demand"],
)

In [28]:
production_yield_model = (
    build_ridge_product_yield_model()
)

production_yield_model.fit(
    prepare_product_share_features(
        historical_product_revenue
    ),
    historical_product_revenue[
        "target_product_net_yield"
    ].astype(float),
)

future_predicted_product_yield = (
    production_yield_model.predict(
        prepare_product_share_features(
            future_mix_features
        )
    )
)

future_product_predictions[
    "predicted_product_net_yield"
] = np.maximum(
    future_predicted_product_yield,
    0.01,
)

future_product_predictions[
    "product_yield_model_name"
] = "ridge_product_yield"

future_product_predictions[
    "predicted_product_net_revenue"
] = (
    future_product_predictions[
        "predicted_product_demand"
    ]
    * future_product_predictions[
        "predicted_product_net_yield"
    ]
)

assert future_product_predictions[
    "predicted_product_net_revenue"
].notna().all()

assert (
    future_product_predictions[
        "predicted_product_net_revenue"
    ] > 0
).all()

In [29]:
future_revenue_forecast = (
    future_product_predictions.groupby(
        [
            "forecast_created_date",
            "target_date",
            "horizon_days",
        ],
        as_index=False,
    )
    .agg(
        predicted_demand=(
            "predicted_demand",
            "first",
        ),
        predicted_net_revenue=(
            "predicted_product_net_revenue",
            "sum",
        ),
    )
)

future_revenue_forecast[
    "revenue_model_name"
] = (
    "sarimax_demand_hgb_mix_"
    "ridge_product_yield"
)

assert len(future_revenue_forecast) == 30

In [30]:
production_revenue_calibration = (
    ridge_yield_revenue_predictions.copy()
)

production_revenue_calibration[
    "horizon_bucket"
] = pd.cut(
    production_revenue_calibration[
        "horizon_days"
    ],
    bins=[0, 7, 14, 21, 30],
    labels=REVENUE_HORIZON_LABELS,
)

production_revenue_calibration[
    "absolute_error"
] = (
    production_revenue_calibration[
        "actual_net_revenue"
    ]
    - production_revenue_calibration[
        "predicted_net_revenue"
    ]
).abs()

future_revenue_forecast[
    "horizon_bucket"
] = pd.cut(
    future_revenue_forecast[
        "horizon_days"
    ],
    bins=[0, 7, 14, 21, 30],
    labels=REVENUE_HORIZON_LABELS,
)

future_revenue_forecast["lower_bound"] = np.nan
future_revenue_forecast["upper_bound"] = np.nan

future_revenue_forecast[
    "interval_confidence"
] = 1 - REVENUE_INTERVAL_ALPHA

future_revenue_forecast[
    "interval_method"
] = "horizon_bucket_conformal"

future_revenue_forecast[
    "calibration_observations"
] = 0

for horizon_bucket in REVENUE_HORIZON_LABELS:
    calibration_errors = (
        production_revenue_calibration.loc[
            production_revenue_calibration[
                "horizon_bucket"
            ].eq(horizon_bucket),
            "absolute_error",
        ]
    )

    quantile = calculate_conformal_quantile(
        calibration_errors
    )

    target_rows = (
        future_revenue_forecast[
            "horizon_bucket"
        ].eq(horizon_bucket)
    )

    point_forecast = (
        future_revenue_forecast.loc[
            target_rows,
            "predicted_net_revenue",
        ]
    )

    future_revenue_forecast.loc[
        target_rows,
        "lower_bound",
    ] = np.maximum(
        point_forecast - quantile,
        0,
    )

    future_revenue_forecast.loc[
        target_rows,
        "upper_bound",
    ] = point_forecast + quantile

    future_revenue_forecast.loc[
        target_rows,
        "calibration_observations",
    ] = len(calibration_errors)

assert future_revenue_forecast[
    [
        "lower_bound",
        "upper_bound",
    ]
].notna().all().all()

assert (
    future_revenue_forecast["lower_bound"]
    <= future_revenue_forecast[
        "predicted_net_revenue"
    ]
).all()

assert (
    future_revenue_forecast[
        "predicted_net_revenue"
    ]
    <= future_revenue_forecast["upper_bound"]
).all()

In [31]:
future_revenue_forecast[
    [
        "target_date",
        "horizon_days",
        "predicted_demand",
        "predicted_net_revenue",
        "lower_bound",
        "upper_bound",
        "horizon_bucket",
        "calibration_observations",
    ]
].round(2)

,target_date,horizon_days,predicted_demand,predicted_net_revenue,lower_bound,upper_bound,horizon_bucket,calibration_observations
0,2026-01-01,1,1591.95,195655.50,173059.32,218251.68,Days 1-7,84
1,2026-01-02,2,1572.14,192144.28,169548.10,214740.46,Days 1-7,84
2,2026-01-03,3,1750.00,207138.60,184542.42,229734.78,Days 1-7,84
3,2026-01-04,4,1515.34,183571.08,160974.90,206167.26,Days 1-7,84
4,2026-01-05,5,1004.06,106334.46,83738.28,128930.64,Days 1-7,84
5,2026-01-06,6,949.74,99678.81,77082.63,122274.99,Days 1-7,84
6,2026-01-07,7,958.93,103413.10,80816.92,126009.28,Days 1-7,84
7,2026-01-08,8,1102.72,115195.49,98422.23,131968.74,Days 8-14,84
8,2026-01-09,9,1323.02,138109.10,121335.84,154882.35,Days 8-14,84
9,2026-01-10,10,1773.20,206348.69,189575.44,223121.95,Days 8-14,84
